<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/01_fatura_dataset_inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import numpy as np
import pandas as pd
from PIL import Image
import os
import json
from torch.utils.data import Dataset

In [14]:
!wget -c "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1" -O /content/FATURA2.zip

--2026-10-09 17:38:18--  https://zenodo.org/records/10371464/files/FATURA2.zip?download=1
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.185.43.153, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 416 REQUESTED_RANGE_NOT_SATISFIABLE

    The file is already fully retrieved; nothing to do.



In [15]:
!unzip -q /content/FATURA2.zip -d /content/FATURA

replace /content/FATURA/invoices_dataset_final/colored_images/Template3_Instance12.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [16]:
print(os.listdir("/content/FATURA"))

['invoices_dataset_final']


In [17]:
base_path = "/content/FATURA/invoices_dataset_final"
print(os.listdir(base_path))

['Strat2_Split.txt', 'images', 'strat1_dev.csv', 'strat1_test.csv', 'strat1_train.csv', 'Annotations', 'colored_images']


In [18]:
train  = pd.read_csv(f'{base_path}/strat1_train.csv')
val = pd.read_csv(f'{base_path}/strat1_dev.csv')
test = pd.read_csv(f"{base_path}/strat1_test.csv")


In [19]:
print("Shape of Train Data:",train.shape)
print("Shape of Val Data:",val.shape)
print("Shape of Test Data:",test.shape)
print()
print()
train.head()

Shape of Train Data: (7500, 2)
Shape of Val Data: (1250, 2)
Shape of Test Data: (1250, 2)




,img_path,annot_path
0,Template1_Instance0.jpg,Template1_Instance0.json
1,Template1_Instance1.jpg,Template1_Instance1.json
2,Template1_Instance2.jpg,Template1_Instance2.json
3,Template1_Instance3.jpg,Template1_Instance3.json
4,Template1_Instance4.jpg,Template1_Instance4.json


In [20]:
os.listdir('/content/FATURA/invoices_dataset_final/Annotations')

['.virtual_documents',
 'Original_Format',
 'layoutlm_HF_format',
 'COCO_compatible_format']

In [21]:
train.head()

,img_path,annot_path
0,Template1_Instance0.jpg,Template1_Instance0.json
1,Template1_Instance1.jpg,Template1_Instance1.json
2,Template1_Instance2.jpg,Template1_Instance2.json
3,Template1_Instance3.jpg,Template1_Instance3.json
4,Template1_Instance4.jpg,Template1_Instance4.json


In [22]:
def template_ids(data):
    for temp_data in data.columns:
        values = data[temp_data].astype(str)
        template = values.str.extract(r"Template(\d+)")[0]

        print(temp_data)
        print(template.dropna().head())
template_ids(train)

img_path
0    1
1    1
2    1
3    1
4    1
Name: 0, dtype: object
annot_path
0    1
1    1
2    1
3    1
4    1
Name: 0, dtype: object


In [23]:
img = Image.open(
    f'{base_path}/images/Template1_Instance0.jpg'
)
print(img.size)
print(img.mode)

(595, 841)
RGB


In [25]:
class FATURADataset(Dataset):
    def __init__(self, dataframe, base_path):
        self.dataframe = dataframe
        self.base_path = base_path

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        sample = self.dataframe.iloc[index]

        return sample


# Making Dataset as object

train_dataset = FATURADataset(train, base_path)
val_dataset = FATURADataset(val, base_path)
test_dataset = FATURADataset(test, base_path)

# Check lengths
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

# Get one sample
sample = train_dataset[1]

print("\nSample:")
print(sample)

Train: 7500
Validation: 1250
Test: 1250

Sample:
img_path       Template1_Instance1.jpg
annot_path    Template1_Instance1.json
Name: 1, dtype: object


In [27]:
train['template_id']=train['img_path'].str.extract(r'Template(\d+)')[0]

In [28]:
print(train['template_id'])

0        1
1        1
2        1
3        1
4        1
        ..
7495    50
7496    50
7497    50
7498    50
7499    50
Name: template_id, Length: 7500, dtype: object


In [30]:
overlap_id = set(train['template_id']) & set(train['template_id'])
print(overlap_id)

{'34', '37', '22', '19', '50', '29', '45', '39', '2', '10', '14', '32', '25', '31', '46', '21', '47', '35', '6', '38', '9', '3', '24', '5', '23', '41', '36', '48', '12', '33', '4', '27', '49', '30', '13', '44', '43', '42', '26', '15', '11', '8', '20', '18', '1', '16', '7', '40', '28', '17'}
